# 커스텀 프로젝트 만들기

- GLUE dataset의 mnli task를 수행하는 프로젝트
- BERT가 아닌 다른 모델 사용
- 선택한 모델의 _tokenizer_와 task 등 적절히 찾아넣기

In [14]:
#라이브러리 버전 확인
import tensorflow
import numpy
import transformers
import argparse

print(tensorflow.__version__)
print(numpy.__version__)
print(transformers.__version__)
print(argparse.__version__)

2.6.0
1.21.4
4.11.3
1.1


In [31]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, BertForSequenceClassification, AdamW
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm.notebook import tqdm

In [32]:
# GPU 설정
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f'Using device: {device}')

Using device: cuda


# 1. mnli 데이터셋 분석해보기

In [33]:
# MNLI 데이터셋 로드
mnli_dataset = load_dataset('glue', 'mnli')

Reusing dataset glue (/aiffel/.cache/huggingface/datasets/glue/mnli/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)


  0%|          | 0/5 [00:00<?, ?it/s]

In [34]:
# 데이터셋 구조 확인
print(mnli_dataset)

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9832
    })
    test_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9796
    })
    test_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9847
    })
})


In [35]:
# train 데이터셋의 첫 번째 샘플 확인
print(mnli_dataset['train'][0])

{'hypothesis': 'Product and geography are what make cream skimming work. ', 'premise': 'Conceptually cream skimming has two basic dimensions - product and geography.', 'label': 1, 'idx': 0}


In [36]:
# 데이터셋에서 100개 샘플씩만 사용
train_subset = Subset(mnli_dataset['train'], range(100))
validation_subset = Subset(mnli_dataset['validation_matched'], range(100))

# 2. MNLIProcessor 클래스 구현하기

In [37]:
# MNLIProcessor 클래스 구현
class MNLIProcessor:
    def __init__(self, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def process(self, examples):
        return self.tokenizer(
            examples['premise'], 
            examples['hypothesis'], 
            truncation=True, 
            padding='max_length', 
            max_length=self.max_length,
            return_tensors="pt"
        )

In [38]:
# Hugging Face tokenizer 로드
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

In [39]:
# MNLIProcessor 인스턴스 생성
processor = MNLIProcessor(tokenizer)

# 3. 위에서 구현한 processor 및 Huggingface에서 제공하는 tokenizer 활용하여 데이터셋 구성하기

In [40]:
# MNLIDataset 클래스 구현
class MNLIDataset(Dataset):
    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        processed = self.processor.process(item)
        return {
            'input_ids': processed['input_ids'].squeeze(0),
            'attention_mask': processed['attention_mask'].squeeze(0),
            'labels': torch.tensor(item['label'])
        }

In [41]:
# 데이터셋 및 DataLoader 생성 (배치 크기 조정 및 num_workers 설정)
batch_size = 16  # 배치 크기 조정

In [42]:
train_dataset = MNLIDataset(train_subset, processor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

validation_dataset = MNLIDataset(validation_subset, processor)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size)

# 4. model을 생성하여 학습 및 테스트 진행


In [43]:
# 모델 생성 및 학습, 테스트 진행
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)
model.to(device)
model.train()

# 옵티마이저 설정
optimizer = AdamW(model.parameters(), lr=2e-5)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at

In [44]:
# 학습 루프 (샘플링된 데이터로 3 epoch 학습)
epochs = 3
for epoch in range(epochs):
    total_loss = 0
    for batch in tqdm(train_loader):
        optimizer.zero_grad()
        
        # 데이터를 GPU로 이동
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # 모델 학습
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    print(f"Epoch {epoch + 1}/{epochs} completed with average loss: {total_loss / len(train_loader)}")

  0%|          | 0/7 [00:00<?, ?it/s]

Epoch 1/3 completed with average loss: 1.138138975415911


  0%|          | 0/7 [00:00<?, ?it/s]

Epoch 2/3 completed with average loss: 1.0392849275044032


  0%|          | 0/7 [00:00<?, ?it/s]

Epoch 3/3 completed with average loss: 0.984132638999394


In [45]:
# 테스트 예시 (작은 배치로 테스트)
model.eval()
total_val_loss = 0
with torch.no_grad():
    for batch in validation_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_val_loss += outputs.loss.item()

    avg_val_loss = total_val_loss / len(validation_loader)
    print(f"Validation Loss: {avg_val_loss}")

Validation Loss: 1.1040959698813302
